In [ ]:

import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import BertForSequenceClassification, PreTrainedTokenizerFast
from pathlib import Path
import glob
import sys
import os

# --- KONFIGURACJA ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {DEVICE}")

# 1. Znajdź najnowszy model
import paths_config
paths_config.ensure_dirs()
triage_models = sorted([d for d in glob.glob(str(paths_config.MODELS_TRIAGE / "bert_triage_model_*")) if Path(d).is_dir()], reverse=True)
if not triage_models:
    raise FileNotFoundError("Nie znaleziono modelu bert_triage_model_* w models/triage/. Uruchom najpierw notebook 05.")

MODEL_PATH = None
for model_dir in triage_models:
    if (Path(model_dir) / "tokenizer.json").exists():
        MODEL_PATH = model_dir
        break

if MODEL_PATH is None:
    raise FileNotFoundError("Nie znaleziono modelu z tokenizerem.")

print(f"📂 Ładowanie modelu z: {MODEL_PATH}")

# 2. Ładowanie Mapowania Etykiet
# Szukamy label_mapping.py
current_dir = Path(os.getcwd())
if (current_dir / "label_mapping.py").exists():
    sys.path.insert(0, str(current_dir))
elif (current_dir / "notebooks_ethos" / "label_mapping.py").exists():
    sys.path.insert(0, str(current_dir / "notebooks_ethos"))
else:
    # Try parent
    sys.path.insert(0, str(current_dir.parent))

try:
    from label_mapping import ID_TO_LABEL, NUM_LABELS, TARGET_NAMES
    LABEL_MAP = ID_TO_LABEL
    print(f"✅ Załadowano mapowanie dla {NUM_LABELS} klas.")
except ImportError:
    print("⚠️ Nie udało się zaimportować label_mapping. Używam domyślnych.")
    NUM_LABELS = 8
    LABEL_MAP = {i: f"CLASS_{i}" for i in range(8)}

# 3. Ładowanie Tokenizera i Modelu
try:
    tokenizer = PreTrainedTokenizerFast.from_pretrained(MODEL_PATH, local_files_only=True)
except:
    from tokenizers import Tokenizer
    tokenizer_obj = Tokenizer.from_file(str(Path(MODEL_PATH) / "tokenizer.json"))
    tokenizer = PreTrainedTokenizerFast(tokenizer_object=tokenizer_obj)

model = BertForSequenceClassification.from_pretrained(
    MODEL_PATH,
    output_attentions=True,
    local_files_only=True,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    ignore_mismatched_sizes=True
).to(DEVICE)
model.eval()
print("✅ Model i Tokenizer gotowe.")


In [ ]:

# --- FUNKCJE POMOCNICZE ---
import numpy as np

def predict_with_explanation(text_seq, threshold=0.3):
    inputs = tokenizer(text_seq, return_tensors="pt").to(DEVICE)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Sigmoid for multi-label; keep all 8 probabilities for display
    probs = torch.sigmoid(outputs.logits)[0]
    all_probs = probs.cpu().numpy()  # shape (8,) for all classes
    
    # Labels above threshold (for print / highlighting)
    results = []
    active_indices = (probs > threshold).nonzero(as_tuple=True)[0]
    if len(active_indices) == 0:
        top_prob, top_idx = torch.max(probs, dim=0)
        results.append((LABEL_MAP[top_idx.item()], top_prob.item()))
    else:
        for idx in active_indices:
            results.append((LABEL_MAP[idx.item()], probs[idx].item()))
    results.sort(key=lambda x: x[1], reverse=True)
    
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    last_layer_attn = outputs.attentions[-1][0]
    avg_attn = last_layer_attn.mean(dim=0).cpu().numpy()
    # Aggregated attention over all layers and heads for publication figure
    all_layers_attn = torch.stack([a[0] for a in outputs.attentions], dim=0)
    agg_attn = all_layers_attn.mean(dim=(0, 1)).cpu().numpy()
    
    return results, tokens, avg_attn, all_probs, agg_attn

def attention_to_token_scores(attn_map):
    # Suma uwagi, jaką dany token otrzymał od innych (kolumny) lub nadał (wiersze)
    # Tutaj patrzymy na średnią uwagę skupioną na tokenie
    return attn_map.mean(axis=0)

def plot_token_scores(tokens, scores, title, predictions_str=None, out_path=None):
    valid_indices = [i for i, t in enumerate(tokens) if t not in ['[CLS]', '[SEP]', '[PAD]']]
    valid_tokens = [tokens[i] for i in valid_indices]
    valid_scores = scores[valid_indices]

    if len(valid_scores) == 0:
        return

    if valid_scores.max() > 0:
        valid_scores = valid_scores / valid_scores.max()
    valid_scores = valid_scores.reshape(-1, 1)

    # Same figsize as notebook display (min height 3 so nephro is not too thin)
    n_tokens = len(valid_tokens)
    fig, ax = plt.subplots(figsize=(1.4, max(3, 0.28 * n_tokens)))
    sns.heatmap(
        valid_scores,
        yticklabels=valid_tokens,
        xticklabels=[],
        cmap="Reds",
        annot=True,
        fmt=".2f",
        cbar=False,
        ax=ax
    )
    ax.set_title(title, fontsize=9)
    if predictions_str:
        plt.tight_layout(rect=[0, 0.18, 1, 1])  # leave bottom 18% for "Model output"
        fig.text(0.5, 0.08, predictions_str, ha='center', fontsize=7)
    else:
        plt.tight_layout()
    if out_path is not None:
        plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
        plt.close()
        return
    plt.savefig('outputs/plots/06_Explainability_cell_1.png', bbox_inches='tight')
    plt.show()


def plot_output_bars(all_probs, class_names, expected_labels, out_path, threshold=0.15, display_names=None):
    """Bar chart of class probabilities (notebook-style). display_names: optional English labels for y-axis."""
    expected_set = {s.strip().upper() for s in expected_labels.split(",")}
    def norm(c):
        return c.upper().replace("TARGET_SPEC_", "").replace("TARGET_POZ", "POZ")
    gt = [norm(c) in expected_set for c in class_names]
    labels = display_names if display_names is not None else class_names
    fig, ax = plt.subplots(figsize=(4.5, 3.5))
    y_pos = np.arange(len(class_names))[::-1]
    colors = ["#2d6a4f" if gt[i] else "#495057" for i in range(len(class_names))]
    ax.barh(y_pos, all_probs, height=0.7, color=colors, edgecolor="none")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("Probability", fontsize=8)
    ax.set_xlim(0, 1.05)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()


In [ ]:

print("\n🔬 ROZPOCZYNAM TESTY NA RZECZYWISTYCH DANYCH Z WALIDACJI (MULTI-LABEL)...\n")

real_patients = [
    {
        "desc": "Complex case (5 labels: Haem/Nefro/SOR/Cardio/Pulmo)",
        "text": "AGE_70 SEX_F HGB_Q1_CRITICAL HCT_Q1_CRITICAL PLT_Q3 MCV_Q5 WBC_Q9 CREATININE_Q9 UREA_Q9 ALT_Q4 AST_VAL COMPOSITE_ANEMIA_INDICATOR COMPOSITE_THROMBOCYTOPENIA_INDICATOR COMPOSITE_RENAL_FAILURE_INDICATOR COMPOSITE_BLEEDING_RISK COMPOSITE_INFECTION_INDICATOR COMPOSITE_SEVERE_ANEMIA SYMPTOM_GI_BLEED_NO HIST_BLOOD_THINNERS_NO SYMPTOM_WEIGHT_LOSS_NO SYMPTOM_CHEST_PAIN_NO SYMPTOM_DYSPNEA_NO",
        "expected": "HEMATOLOG, NEFROLOG, SOR, KARDIOLOG, PULMONOLOG"
    },
    {
        "desc": "Nephrology only (Age 60, Creatinine High)",
        "text": "AGE_60 SEX_M HGB_Q5 HCT_Q5 PLT_Q6 MCV_Q5 WBC_Q5 CREATININE_Q9 ALT_Q5",
        "expected": "NEFROLOG"
    },
    {
        "desc": "Cardio vs Pulmo confusion (chest pain)",
        "text": "AGE_60 SEX_F HGB_Q2 HCT_Q3 PLT_Q4 MCV_Q4 WBC_Q4 CREATININE_Q6 UREA_Q8 COMPOSITE_ANEMIA_INDICATOR SYMPTOM_GI_BLEED_NO SYMPTOM_WEIGHT_LOSS_NO SYMPTOM_CHEST_PAIN_YES",
        "expected": "KARDIOLOG, PULMONOLOG, HEMATOLOG"
    }
]

for p in real_patients:
    print("\n" + "="*60)
    print(f"--- Case: {p['desc']} ---")
    print(f"Input sequence: {p['text']}")
    print(f"Expected labels: {p['expected']}")
    
    # Predict (returns all 8 class probs + aggregated attention over layers/heads)
    predictions, tokens, attn_map, all_probs, agg_attn = predict_with_explanation(p['text'], threshold=0.15)
    
    print("Model output (threshold 15%):")
    if not predictions:
        print("  No predictions above threshold.")
    else:
        for label, conf in predictions:
            mark = "[+]" if conf > 0.5 else "[-]"
            print(f"  {mark} {label}: {conf*100:.1f}%")
    
    # Show all 8 classes on the plot (so e.g. "Nephrology only" shows others low)
    parts = [f"{TARGET_NAMES[i]}: {all_probs[i]:.2f}" for i in range(len(all_probs))]
    predictions_str = "Model output: " + ", ".join(parts[:4]) + "\n" + ", ".join(parts[4:])
    
    scores = attention_to_token_scores(attn_map)
    plot_token_scores(tokens, scores, title=f"Token attribution: {p['desc']}", predictions_str=predictions_str)


In [ ]:
# --- Paper figures: no probabilities subtitle; English specialization names ---
TARGET_NAMES_EN = [
    "Primary care", "Gastroenterology", "Haematology", "Nephrology",
    "Emergency/ICU", "Cardiology", "Pulmonology", "Hepatology"
]
paper_cases = [
    ("complex", real_patients[0], "Token attribution: Complex case (5 referrals)"),
    ("nephro", real_patients[1], "Token attribution: Nephrology only"),
]
out_plots = Path(paths_config.OUTPUTS_PLOTS)
out_plots.mkdir(parents=True, exist_ok=True)
for name, case, title in paper_cases:
    _, tokens, _, all_probs, agg_attn = predict_with_explanation(case["text"], threshold=0.15)
    scores = attention_to_token_scores(agg_attn)
    plot_token_scores(tokens, scores, title=title, out_path=out_plots / f"attention_{name}.png")
    plot_output_bars(all_probs, TARGET_NAMES, case["expected"], out_plots / f"output_{name}.png", display_names=TARGET_NAMES_EN)
print("Paper figures saved:", [f"attention_{n}.png, output_{n}.png" for n, _, _ in paper_cases])

In [ ]:
# Paper uses: attention_complex.png, output_complex.png, attention_nephro.png, output_nephro.png (from cell above)